Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Hilbert Spaces & Fourier, Properly

> ⚠️ **Draft — pending instructor review.** Visuals execute; proofs need a human pass before teaching. Remove this banner after review.

The bridge between [Measure Theory's](../Analysis/Measure_Theory.ipynb) $L^2$ and [DSP's](../../Intro_DSP/README.md) transforms: once you see $L^2$ as a geometry — with angles, projections, and orthonormal bases — 'transform = change of basis' stops being a slogan and becomes a theorem, and Parseval becomes Pythagoras.

## 1. Pre-requisites

- [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S1–S2 (bases, projections in $\mathbb{R}^N$).
- [Measure Theory](../Analysis/Measure_Theory.ipynb) S3 ($L^p$ spaces) for full rigor — skimmable if you accept $L^2$ as 'finite-energy signals'.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

---
### 🕐 Session 1 of 3 — *Inner Products & Orthonormal Systems* (~35 min)
**Goal:** give function spaces a geometry; measure angles between signals.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S1–S2. &nbsp; **Feeds into:** Session 2 (the projection theorem).

---

## 2. Inner Product Spaces

💡 **Intuition.** An inner product is a *dot product with the finite dimensions removed*: $\langle f, g \rangle = \int f \bar{g}$ sums the pointwise agreement of two signals just as $x^Ty$ sums coordinate agreement. Everything geometric follows: length $\|f\| = \sqrt{\langle f, f\rangle}$ (the energy!), angle via Cauchy–Schwarz, orthogonality as zero correlation. A **Hilbert space** is an inner product space that is also *complete* — Cauchy sequences of signals converge ([Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb) upgraded to functions) — and $L^2$ is the star example.

**Cauchy–Schwarz.** $|\langle f, g \rangle| \le \|f\| \|g\|$.

*Proof.* For any $t \in \mathbb{R}$ (real case): $0 \le \|f - t g\|^2 = \|f\|^2 - 2t\langle f, g\rangle + t^2 \|g\|^2$ — a quadratic in $t$ that never goes negative, so its discriminant $4\langle f,g\rangle^2 - 4\|f\|^2\|g\|^2 \le 0$. $\blacksquare$

This is Hölder with $p = q = 2$, and it's why 'correlation coefficient' lands in $[-1, 1]$ — matched filtering in [Statistical SP](../../Intro_DSP/Statistical_Signal_Processing.ipynb) is maximizing this very inner product.

In [2]:
# Angles between signals: correlation as cosine
t = np.linspace(0, 1, 1000)
f = np.sin(2 * np.pi * 5 * t)
for name, g in [("same sine", np.sin(2*np.pi*5*t)), ("shifted 90°", np.cos(2*np.pi*5*t)),
                ("different freq", np.sin(2*np.pi*8*t)), ("sine + noise", np.sin(2*np.pi*5*t) + np.random.default_rng(0).standard_normal(1000))]:
    cos = (f @ g) / (np.linalg.norm(f) * np.linalg.norm(g))
    print(f"cos∠(f, {name:15s}) = {cos:+.3f}")
print("→ orthogonality = zero correlation; harmonics are mutually orthogonal")

cos∠(f, same sine      ) = +1.000
cos∠(f, shifted 90°    ) = +0.000
cos∠(f, different freq ) = +0.000
cos∠(f, sine + noise   ) = +0.611
→ orthogonality = zero correlation; harmonics are mutually orthogonal


---
### 🕐 Session 2 of 3 — *The Projection Theorem* (~35 min)
**Goal:** prove that closed subspaces admit unique best approximations; get least squares for signals.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Fourier bases).

---

## 3. Best Approximation

💡 **Intuition.** Same shadow picture as [Linear Algebra S2](../Linear_Algebra/Linear_Algebra.ipynb) — but in infinite dimensions the *existence* of the closest point is no longer free: you need completeness to stop minimizing sequences from converging to a hole. That's the real job of the 'Hilbert' in Hilbert space.

**Theorem (projection).** Let $M$ be a closed subspace of a Hilbert space $H$, $f \in H$. Then there is a *unique* $\hat{f} \in M$ minimizing $\|f - m\|$ over $m \in M$, characterized by $f - \hat{f} \perp M$.

*Proof sketch.* Take a minimizing sequence $m_n$ with $\|f - m_n\| \to d = \inf$. The **parallelogram law** $\|a+b\|^2 + \|a-b\|^2 = 2\|a\|^2 + 2\|b\|^2$ applied to $a = f - m_n$, $b = f - m_k$ shows $\|m_n - m_k\|^2 \le 2\|f-m_n\|^2 + 2\|f-m_k\|^2 - 4d^2 \to 0$ (using $\frac{m_n + m_k}{2} \in M$) — the sequence is Cauchy, and **completeness** hands us the limit $\hat f \in M$ ($M$ closed). Orthogonality: if $\langle f - \hat f, m\rangle \ne 0$ for some unit $m \in M$, then $\hat f + \langle f - \hat f, m\rangle m$ is strictly closer — contradiction. Uniqueness follows from Pythagoras. $\blacksquare$

**Payoff.** For an orthonormal system $\{e_k\}$, the projection onto $\mathrm{span}\{e_1..e_K\}$ is $\hat f = \sum_{k\le K} \langle f, e_k\rangle e_k$ — *keep the top coefficients* — with error $\|f\|^2 - \sum_{k \le K} |\langle f, e_k\rangle|^2$ (**Bessel**: partial sums of coefficient energy never exceed the signal's energy). Every truncated Fourier/wavelet approximation in DSP is this theorem running.

In [3]:
# Best L² approximation of a square wave by K harmonics — projection in action
t = np.linspace(0, 1, 4000, endpoint=False)
sq = np.sign(np.sin(2 * np.pi * t))

plt.figure(figsize=(8.5, 3))
plt.plot(t, sq, "k", linewidth=1, label="square wave")
for K, alpha in [(1, 0.5), (5, 0.7), (25, 1.0)]:
    approx = np.zeros_like(t)
    for k in range(1, K + 1, 2):                       # odd harmonics only
        approx += (4 / (np.pi * k)) * np.sin(2 * np.pi * k * t)
    plt.plot(t, approx, alpha=alpha, label=f"K={K} harmonics")
plt.legend(); plt.title("Projections onto growing subspaces (note Gibbs' stubborn ears)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2016431/1099153454.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Orthonormal Bases & Parseval* (~40 min)
**Goal:** define completeness of a basis; prove Parseval; see why the Fourier basis is special.
**Builds on:** Session 2.

---

## 4. Complete Orthonormal Bases

An orthonormal system $\{e_k\}$ is a **basis** (complete) if finite combinations are dense — equivalently, if Bessel's inequality is *equality* for every $f$:

$$\|f\|^2 = \sum_k |\langle f, e_k \rangle|^2 \qquad \textbf{(Parseval)}$$

💡 **Intuition.** Parseval is *Pythagoras with infinitely many perpendicular directions*: energy in the signal = summed energy of its coordinates, because orthogonal components can't interfere. 'The DFT/Fourier transform preserves energy' is not a happy accident — it is the statement that complex exponentials form a complete orthonormal basis of $L^2$ (Riesz–Fischer; completeness proof via Stone–Weierstrass or Fejér, stated not proved).

**Why the Fourier basis, of all bases?** The exponentials $e^{i\omega t}$ are the **eigenfunctions of time-invariant systems**: feed $e^{i\omega t}$ into any LTI filter and you get $H(\omega) e^{i\omega t}$ — same signal, scaled ([Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb)'s eigenvectors, in function space). The Fourier basis simultaneously diagonalizes *every* convolution — which is why convolution becomes multiplication and DSP lives in the frequency domain.

In [4]:
# Parseval, numerically, with the DFT (unitary convention)
rng = np.random.default_rng(1)
x = rng.standard_normal(1024)
X = np.fft.fft(x) / np.sqrt(1024)
print(f"time-domain energy   {np.sum(x**2):.6f}")
print(f"coefficient energy   {np.sum(np.abs(X)**2):.6f}   (Parseval: equal)")

# And the eigenfunction property: convolution acts diagonally on exponentials
h = np.exp(-np.arange(30) / 5.0)                       # some LTI filter
k = 37                                                  # pick a frequency bin
e = np.exp(2j * np.pi * k * np.arange(1024) / 1024)    # basis exponential
y = np.convolve(e, h)[:1024]                            # (circular edge effects negligible mid-signal)
ratio = y[200:800] / e[200:800]
print(f"filter output / input on e_k: constant ≈ {ratio.mean():.4f} (std {ratio.std():.2e}) = H(ω_k)")

time-domain energy   1009.508255
coefficient energy   1009.508255   (Parseval: equal)
filter output / input on e_k: constant ≈ 2.6988-2.4525j (std 1.90e-14) = H(ω_k)


## 5. Conclusion

$L^2$ is geometry: Cauchy–Schwarz gives angles, completeness + the parallelogram law give unique best approximations, and Parseval is Pythagoras. The Fourier basis earns its throne by diagonalizing every LTI system at once.

---
## Where next

- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — re-read Sessions 3–5 with this geometry in mind.
- [Kernel Methods & RKHS](../../Intro_Mach_Learn/Kernel_Methods.ipynb) — Hilbert spaces where evaluation is an inner product: the ML sequel.
- [Foundations of Signal Processing 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — wavelets: a *different* orthonormal basis with different trade-offs.